# BigQuery Graph for GraphRAG

> [BigQuery](https://cloud.google.com/bigquery) is a fully managed, AI-ready data analytics platform that helps you maximize value from your data. BigQuery supports [property graphs](https://cloud.google.com/bigquery/docs/graph-overview), enabling graph-based analysis and retrieval.

This notebook demonstrates how to use `BigQuery Graph` for **GraphRAG** with the custom retriever `BigQueryGraphVectorContextRetriever` and compares the response of GraphRAG with conventional RAG.

This notebook is adapted from [Spanner Graph RAG](https://github.com/googleapis/langchain-google-spanner-python/blob/main/docs/graph_rag.ipynb) with modifications for BigQuery Graph.

## Library Installation

Install `langchain-bigquery-graph` and required packages.

In [1]:
%%sh

pip freeze | grep -E "langchain|bigquery|spanner-graph|pandas|pydantic"

bigquery-magics==0.14.0
google-cloud-bigquery==3.41.0
langchain==1.2.15
langchain-bigquery-graph @ git+https://github.com/ksmin23/langchain-bigquery-graph.git@21f4f0f2ae7671c230027a6617e88dc4f6f0397d
langchain-classic==1.0.4
langchain-community==0.4.1
langchain-core==1.3.2
langchain-experimental==0.4.1
langchain-google-genai==4.2.2
langchain-google-vertexai==3.2.3
langchain-protocol==0.0.13
langchain-tests==1.1.7
langchain-text-splitters==1.1.2
pandas==2.3.3
pydantic==2.13.3
pydantic-settings==2.14.0
pydantic_core==2.46.3
spanner-graph-notebook==1.1.10


In [2]:
%%sh

pip install "langchain-bigquery-graph[examples] @ git+https://github.com/ksmin23/langchain-bigquery-graph.git" --quiet
pip install "bigquery-magics[spanner-graph-notebook]" --quiet

In [3]:
# Clear Python's internal import finder caches so that newly installed packages
# can be discovered without restarting the kernel.
# This is a lightweight alternative to kernel restart: it invalidates stale
# path entries cached by importlib's finders, allowing subsequent `import`
# statements to pick up packages that were installed after the interpreter started.
import importlib

importlib.invalidate_caches()

## Authentication

Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

If you are using Vertex AI Workbench, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

## Set Your Google Cloud Project

In [4]:
PROJECT_ID = !gcloud config get project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

PROJECT_ID, LOCATION

## API Enablement

The `langchain-bigquery-graph` package requires that you [enable the BigQuery API](https://console.cloud.google.com/flows/enableapi?apiid=bigquery.googleapis.com) in your Google Cloud Project.

In [5]:
import subprocess

result = subprocess.run(
    ["gcloud", "services", "enable", "bigquery.googleapis.com",
     "--project", PROJECT_ID],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("BigQuery API enabled successfully.")
else:
    print(f"{result.stderr.strip()}")

BigQuery API enabled successfully.


## Set BigQuery Values

Find your dataset values in the [BigQuery Console](https://console.cloud.google.com/bigquery).

**NOTE:**
- The BigQuery dataset must be created beforehand.
- The property graph does NOT need to be created beforehand.
  If such a graph already exists, graphs will be built atop; otherwise a new graph is automatically created.

In [6]:
# @title Set Your Values Here { display-mode: "form" }
DATASET_ID = "graph_rag_demo"  # @param {type: "string"}
GRAPH_NAME = "retail_graph"  # @param {type: "string"}

### Create BigQuery Dataset

The dataset must exist before using `BigQueryGraphStore`. Tables and property graphs are created automatically.

In [7]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION

try:
    bq_client.create_dataset(dataset)
    print(f"Dataset {PROJECT_ID}:{DATASET_ID} successfully created.")
except Exception as e:
    print(f"{e}")

## BigQueryGraphStore

To initialize the `BigQueryGraphStore` class you need to provide:

1. A Google Cloud project ID
2. A BigQuery dataset ID
3. A property graph name to create in the dataset

Other arguments are optional and only need to be passed if different from defaults.

In [8]:
from langchain_bigquery_graph import BigQueryGraphStore

graph_store = BigQueryGraphStore(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    graph_name=GRAPH_NAME,
    location=LOCATION,
)

## Add Graph Documents

Add graph documents to the graph store.

### Load Documents

In [9]:
!wget https://raw.githubusercontent.com/googleapis/langchain-google-spanner-python/main/samples/retaildata.zip -P content
!unzip "content/retaildata.zip" -d content

--2026-05-01 07:11:53--  https://raw.githubusercontent.com/googleapis/langchain-google-spanner-python/main/samples/retaildata.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60031 (59K) [application/zip]
Saving to: ‘content/retaildata.zip’

retaildata.zip      100%[===================>]  58.62K  --.-KB/s    in 0.01s   

2026-05-01 07:11:54 (5.34 MB/s) - ‘content/retaildata.zip’ saved [60031/60031]

Archive:  content/retaildata.zip
   creating: content/retaildata/
   creating: content/retaildata/0/
  inflating: content/retaildata/0/5.txt  
  inflating: content/retaildata/0/4.txt  
  inflating: content/retaildata/0/6.txt  
  inflating: content/retaildata/0/7.txt  
  inflating: content/retaildata/0/3.txt  
  inflating: content/retaildata/0/2.txt  
  inflati

In [10]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

path = "content/retaildata/"
directories = [
    item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))
]

document_lists = []
for directory in directories:
    loader = DirectoryLoader(
        os.path.join(path, directory), glob="**/*.txt", loader_cls=TextLoader
    )
    document_lists.append(loader.load())

In [11]:
len(document_lists)

6

### Extract Nodes and Edges

In [12]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Category", "Segment", "Tag", "Product", "Bundle", "Deal"],
    allowed_relationships=[
        "In_Category",
        "Tagged_With",
        "In_Segment",
        "In_Bundle",
        "Is_Accessory_Of",
        "Is_Upgrade_Of",
        "Has_Deal",
    ],
    node_properties=[
        "name",
        "price",
        "weight",
        "deal_end_date",
        "features",
    ],
)

/tmp/ipykernel_24063/249239270.py:4: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)
/tmp/ipykernel_24063/249239270.py:4: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)


In [13]:
graph_documents = []
for document_list in document_lists:
    graph_documents.extend(llm_transformer.convert_to_graph_documents(document_list))

In [14]:
len(graph_documents)

68

In [15]:
# Add embeddings to the graph documents for Product nodes
embedding_service = VertexAIEmbeddings(model_name="gemini-embedding-001")
for graph_document in graph_documents:
    for node in graph_document.nodes:
        if "features" in node.properties:
            node.properties["embedding"] = embedding_service.embed_query(
                node.properties["features"]
            )

/tmp/ipykernel_24063/4160170288.py:2: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_service = VertexAIEmbeddings(model_name="gemini-embedding-001")
/tmp/ipykernel_24063/4160170288.py:2: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embedding_service = VertexAIEmbeddings(model_name="gemini-embedding-001")


In [16]:
import copy


def print_graph(graph_documents):
    for doc in graph_documents:
        print(f"{doc.source.page_content[:100]} === truncated ===")
        nodes = copy.deepcopy(doc.nodes)
        for node in nodes:
            if "embedding" in node.properties:
                node.properties["embedding"] = "..."
        print(nodes)
        print(doc.relationships)
        print()

In [17]:
print_graph(graph_documents)


  Product: SkyHawk Zephyr Propeller Guards
  Price: $14.99
  Weight: 20g (0.7 oz)
  Fly with Confid === truncated ===
[Node(id='Skyhawk Zephyr Propeller Guards', type='Product', properties={'name': 'SkyHawk Zephyr Propeller Guards', 'price': '$14.99', 'weight': '20g (0.7 oz)', 'features': 'Enhanced Safety: Prevent accidental damage to the propellers and surroundings during flight. Lightweight Design: Minimal impact on flight performance. Easy Installation: Snap on and off in seconds for quick and convenient use. Increased Durability: Provides an extra layer of protection for your drone.', 'embedding': '...'}), Node(id='Skyhawk Zephyr Drone', type='Product', properties={'name': 'SkyHawk Zephyr Drone'}), Node(id='Photography', type='Tag', properties={'name': 'Photography'}), Node(id='Videography', type='Tag', properties={'name': 'Videography'})]
[Relationship(source=Node(id='Skyhawk Zephyr Propeller Guards', type='Product', properties={}), target=Node(id='Skyhawk Zephyr Drone', type='Pr

### Post-process Extracted Nodes and Edges

Apply your domain knowledge to clean up and make desired fixes to the generated graph in the earlier step.

In [18]:
# set of all valid products
products = set()


def prune_invalid_products():
    for graph_document in graph_documents:
        nodes_to_remove = []
        for node in graph_document.nodes:
            if node.type == "Product" and "features" not in node.properties:
                nodes_to_remove.append(node)
            else:
                products.add(node.id)
        for node in nodes_to_remove:
            graph_document.nodes.remove(node)


def prune_invalid_segments(valid_segments):
    for graph_document in graph_documents:
        nodes_to_remove = []
        for node in graph_document.nodes:
            if node.type == "Segment" and node.id not in valid_segments:
                nodes_to_remove.append(node)
        for node in nodes_to_remove:
            graph_document.nodes.remove(node)


def fix_directions(relation_name, wrong_source_type):
    for graph_document in graph_documents:
        for relationship in graph_document.relationships:
            if relationship.type == relation_name:
                if relationship.source.type == wrong_source_type:
                    source = relationship.source
                    target = relationship.target
                    relationship.source = target
                    relationship.target = source


def is_not_a_listed_product(node):
    if node.type == "Product" and node.id not in products:
        return True
    return False


def prune_dangling_relationships():
    for graph_document in graph_documents:
        relationships_to_remove = []
        for relationship in graph_document.relationships:
            if is_not_a_listed_product(
                relationship.source
            ) or is_not_a_listed_product(relationship.target):
                relationships_to_remove.append(relationship)
        for relationship in relationships_to_remove:
            graph_document.relationships.remove(relationship)


def prune_unwanted_relationships(relation_name, source, target):
    node_types = set([source, target])
    for graph_document in graph_documents:
        relationships_to_remove = []
        for relationship in graph_document.relationships:
            if (
                relationship.type == relation_name
                and set([relationship.source.type, relationship.target.type])
                == node_types
            ):
                relationships_to_remove.append(relationship)
        for relationship in relationships_to_remove:
            graph_document.relationships.remove(relationship)


prune_invalid_products()
prune_invalid_segments(set(["Home", "Office", "Fitness"]))
prune_unwanted_relationships("IN_CATEGORY", "Bundle", "Category")
prune_unwanted_relationships("IN_CATEGORY", "Deal", "Category")
prune_unwanted_relationships("IN_SEGMENT", "Bundle", "Segment")
prune_unwanted_relationships("IN_SEGMENT", "Deal", "Segment")
prune_dangling_relationships()
fix_directions("HAS_DEAL", "Deal")
fix_directions("IN_BUNDLE", "Bundle")

print_graph(graph_documents)


  Product: SkyHawk Zephyr Propeller Guards
  Price: $14.99
  Weight: 20g (0.7 oz)
  Fly with Confid === truncated ===
[Node(id='Skyhawk Zephyr Propeller Guards', type='Product', properties={'name': 'SkyHawk Zephyr Propeller Guards', 'price': '$14.99', 'weight': '20g (0.7 oz)', 'features': 'Enhanced Safety: Prevent accidental damage to the propellers and surroundings during flight. Lightweight Design: Minimal impact on flight performance. Easy Installation: Snap on and off in seconds for quick and convenient use. Increased Durability: Provides an extra layer of protection for your drone.', 'embedding': '...'}), Node(id='Photography', type='Tag', properties={'name': 'Photography'}), Node(id='Videography', type='Tag', properties={'name': 'Videography'})]
[Relationship(source=Node(id='Skyhawk Zephyr Propeller Guards', type='Product', properties={}), target=Node(id='Skyhawk Zephyr Drone', type='Product', properties={}), type='IS_ACCESSORY_OF', properties={}), Relationship(source=Node(id='S

### Load Data to BigQuery Graph

In [19]:
# Uncomment the line below, if you want to cleanup from previous iterations.
# BeWARE - THIS COULD REMOVE DATA FROM YOUR DATABASE !!!
graph_store.cleanup()
graph_store.add_graph_documents(graph_documents)

## Verify the Graph with BigQuery Magics

Visualize and verify the ingested graph data using `bigquery_magics`.

In [20]:
%load_ext bigquery_magics

The bigquery_magics extension is already loaded. To reload it, use:
  %reload_ext bigquery_magics


In [21]:
%%bigquery --graph --pyformat --project {PROJECT_ID}
GRAPH `{DATASET_ID}`.`{GRAPH_NAME}`
MATCH (a)-[e]->(b)
RETURN TO_JSON(a) AS a, TO_JSON(e) AS e, TO_JSON(b) AS b

Run GQL queries to verify the ingested data.

In [22]:
# Query all Product nodes
gql = (
    f"GRAPH `{DATASET_ID}`.`{GRAPH_NAME}` "
    "MATCH (p:Product) "
    "RETURN p.id AS product_id, p.price AS price, p.weight AS weight"
)
print(f"GQL: {gql}\n")

results = graph_store.query(gql)
for row in results:
    print(f"  {row}")

GQL: GRAPH `graph_rag_demo`.`retail_graph` MATCH (p:Product) RETURN p.id AS product_id, p.price AS price, p.weight AS weight

  {'product_id': 'Privacyguard Standard', 'price': '49.00', 'weight': None}
  {'product_id': 'Secureview Privacy Screen', 'price': '$79.00', 'weight': None}
  {'product_id': 'Chromapaper Pro Pack', 'price': '49.99', 'weight': None}
  {'product_id': 'Secure Erase Tool', 'price': '$9.99', 'weight': None}
  {'product_id': 'Auraview 32 Monitor', 'price': None, 'weight': None}
  {'product_id': 'Quantum Shield', 'price': None, 'weight': None}
  {'product_id': 'Chromaclean Kit', 'price': '$29.99', 'weight': None}
  {'product_id': 'Quantum Shield Screen Protector', 'price': '$19', 'weight': None}
  {'product_id': 'Ergoflex Pro Standing Desk', 'price': None, 'weight': None}
  {'product_id': 'Aura X5 Smartphone', 'price': '$899.00', 'weight': None}
  {'product_id': 'Sonaris Pro Headphones', 'price': None, 'weight': None}
  {'product_id': 'Quantum Charger', 'price': '$69.0

In [23]:
# Query relationships
gql = (
    f"GRAPH `{DATASET_ID}`.`{GRAPH_NAME}` "
    "MATCH (a:Product)-[r]->(b) "
    "RETURN a.id AS source, LABELS(r) AS relationship, b.id AS target"
)
print(f"GQL: {gql}\n")

results = graph_store.query(gql)
for row in results:
    print(f"  {row}")

GQL: GRAPH `graph_rag_demo`.`retail_graph` MATCH (a:Product)-[r]->(b) RETURN a.id AS source, LABELS(r) AS relationship, b.id AS target

  {'source': 'Lumix Keyboard', 'relationship': ['IN_SEGMENT'], 'target': 'Office'}
  {'source': 'Secureview Privacy Screen', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Aurabook Air'}
  {'source': 'Secureview Privacy Screen', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Aurabook X500'}
  {'source': 'Chromapaper Pro Pack', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Chromajet Pro Xl'}
  {'source': 'Chromapaper Pro Pack', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Chromajet Pro Xl Printer'}
  {'source': 'Secure Erase Tool', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Quantumleap Flash'}
  {'source': 'Secure Erase Tool', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Titandrive X5000'}
  {'source': 'Chromaclean Kit', 'relationship': ['IS_ACCESSORY_OF'], 'target': 'Chromajet Pro Xl'}
  {'source': 'Chromaclean Kit', 'relationship': ['IS

In [24]:
# Display graph schema
print(graph_store.get_schema)

{
  "Name of graph": "graph_rag_demo.retail_graph",
  "Node properties per node label": {
    "Bundle": [
      {
        "name": "embedding",
        "type": "ARRAY<FLOAT64>"
      },
      {
        "name": "features",
        "type": "STRING"
      },
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "name",
        "type": "STRING"
      },
      {
        "name": "price",
        "type": "STRING"
      }
    ],
    "Category": [
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "name",
        "type": "STRING"
      }
    ],
    "Deal": [
      {
        "name": "deal_end_date",
        "type": "STRING"
      },
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "name",
        "type": "STRING"
      },
      {
        "name": "price",
        "type": "STRING"
      }
    ],
    "Product": [
      {
        "name": "embedding",
        "type": "ARRAY<FLOAT64>"
      },


## GraphRAG Flow Using BigQuery Graph

### Question Answering Prompt

In [25]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

from IPython.display import Markdown
import textwrap


def format_docs(docs):
    print("Context Retrieved: \n")
    for doc in docs:
        print(textwrap.fill(doc.page_content, width=80))
        print("\n")

    context = "\n\n".join(doc.page_content for doc in docs)
    return context


BIGQUERY_GRAPH_QA_TEMPLATE = """
You are a helpful and friendly AI assistant for question answering tasks for an electronics
retail online store.
Create a human readable answer for the question.
You should only use the information provided in the context and not use your internal knowledge.
Don't add any information.
Here is an example:

Question: Which funds own assets over 10M?
Context:[name:ABC Fund, name:Star fund]"
Helpful Answer: ABC Fund and Star fund have assets over 10M.

Follow this example when generating answers.
You are given the following information:
- `Question`: the natural language question from the user
- `Graph Schema`: contains the schema of the graph database
- `Graph Query`: A BigQuery Graph GQL query equivalent of the question from the user used to extract context from the graph database
- `Context`: The response from the graph database as context. The context has nodes and edges. Use the relationships.
Information:
Question: {question}
Graph Schema: {graph_schema}
Context: {context}

Format your answer to be human readable.
Use the relationships in the context to answer the question.
Only include information that is relevant to a customer.
Helpful Answer:"""

prompt = PromptTemplate(
    template=BIGQUERY_GRAPH_QA_TEMPLATE,
    input_variables=["question", "graph_schema", "context"],
)

llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

chain = prompt | llm | StrOutputParser()

/tmp/ipykernel_24063/4041453899.py:52: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)


### GraphRAG Using Vector Search and Graph Expansion

**How it works:**
1. Vector search runs as SQL on the base table (BigQuery property graphs don't support ARRAY properties in GQL)
2. Matching node IDs are used to traverse the graph via GQL
3. Results include the matched nodes and their neighbors up to `expand_by_hops` distance

In [26]:
from langchain_bigquery_graph import BigQueryGraphVectorContextRetriever
from langchain_google_vertexai import VertexAIEmbeddings


def use_node_vector_retriever(
    question, graph_store, embedding_service, label_expr, expand_by_hops
):
    retriever = BigQueryGraphVectorContextRetriever.from_params(
        graph_store=graph_store,
        embedding_service=embedding_service,
        label_expr=label_expr,
        expand_by_hops=expand_by_hops,
        top_k=1,
        k=10,
    )
    context = format_docs(retriever.invoke(question))
    return context


embedding_service = VertexAIEmbeddings(model_name="gemini-embedding-001")

/tmp/ipykernel_24063/3577218710.py:20: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_service = VertexAIEmbeddings(model_name="gemini-embedding-001")


In [27]:
import textwrap

question = "Give me recommendations for a beginner drone"
context = use_node_vector_retriever(
    question, graph_store, embedding_service, label_expr="Product", expand_by_hops=1
)

answer = chain.invoke(
    {"question": question, "graph_schema": graph_store.get_schema, "context": context}
)

print("\n\nAnswer:\n")
print(textwrap.fill(answer, width=80))

Context Retrieved: 

[{"kind": "node", "labels": ["Product"], "properties": {"features": "Simple
Controls, Tough Build, Capture Memories, Extended Fun, Worry-Free Flying", "id":
"Skyhawk Zephyr Drone", "name": "SkyHawk Zephyr Drone", "price": "$129.99",
"weight": "220g (7.8 oz)"}}, {"kind": "edge", "labels": ["TAGGED_WITH"],
"properties": {"id": "Skyhawk Zephyr Drone", "target_id": "Videography"}},
{"kind": "node", "labels": ["Tag"], "properties": {"id": "Videography", "name":
"Videography"}}]


[{"kind": "node", "labels": ["Product"], "properties": {"features": "Simple
Controls, Tough Build, Capture Memories, Extended Fun, Worry-Free Flying", "id":
"Skyhawk Zephyr Drone", "name": "SkyHawk Zephyr Drone", "price": "$129.99",
"weight": "220g (7.8 oz)"}}]


[{"kind": "node", "labels": ["Product"], "properties": {"features": "Simple
Controls, Tough Build, Capture Memories, Extended Fun, Worry-Free Flying", "id":
"Skyhawk Zephyr Drone", "name": "SkyHawk Zephyr Drone", "price": "$129.99",
"w

In [28]:
question = "What are the best deals on fitness products?"
context = use_node_vector_retriever(
    question, graph_store, embedding_service, label_expr="Product", expand_by_hops=2
)

answer = chain.invoke(
    {"question": question, "graph_schema": graph_store.get_schema, "context": context}
)

print("\n\nAnswer:\n")
print(textwrap.fill(answer, width=80))

Context Retrieved: 

[{"kind": "node", "labels": ["Product"], "properties": {"features": "Universal
Compatibility, Sturdy Steel Construction, Easy Installation, Scratch-Resistant
Finish, Low Profile Design", "id": "Vesa Mount Adapter", "name": "VESA Mount
Adapter", "price": "$19.99", "weight": "17.6 lbs (8 kg)"}}, {"kind": "edge",
"labels": ["IN_CATEGORY"], "properties": {"id": "Vesa Mount Adapter",
"target_id": "Accessories"}}, {"kind": "node", "labels": ["Category"],
"properties": {"id": "Accessories", "name": null}}]


[{"kind": "node", "labels": ["Product"], "properties": {"features": "Universal
Compatibility, Sturdy Steel Construction, Easy Installation, Scratch-Resistant
Finish, Low Profile Design", "id": "Vesa Mount Adapter", "name": "VESA Mount
Adapter", "price": "$19.99", "weight": "17.6 lbs (8 kg)"}}, {"kind": "edge",
"labels": ["IN_SEGMENT"], "properties": {"id": "Vesa Mount Adapter",
"target_id": "Home"}}, {"kind": "node", "labels": ["Segment"], "properties":
{"id": "Home",

## Compare with Conventional RAG

### Setup and Load Data for Vector Search

Use an in-memory vector store for comparison. The same retail documents are chunked and embedded for vector similarity search.

In [29]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create splits for documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
splits = text_splitter.split_documents(
    [document for document_list in document_lists for document in document_list]
)

# Initialize in-memory vector store and load data
embeddings = VertexAIEmbeddings(model_name="gemini-embedding-001")
vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)

print(f"Loaded {len(splits)} chunks into the vector store.")

/tmp/ipykernel_24063/1349294043.py:13: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(model_name="gemini-embedding-001")


Loaded 72 chunks into the vector store.


### Create a Conventional RAG Chain

In [30]:
from langchain_core.runnables import RunnablePassthrough
import textwrap


def format_docs(docs):
    print("Context Retrieved: \n")
    for doc in docs:
        print(textwrap.fill(doc.page_content, width=80))
        print("\n")

    context = "\n\n".join(doc.page_content for doc in docs)
    return context


rag_prompt = PromptTemplate(
    template="""
    You are a friendly digital shopping assistant.
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Question: {question}
    Context: {context}
    Answer:
  """,
    input_variables=["context", "question"],
)

# Create a RAG chain
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
rag_chain = (
    {
        "context": vector_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

### Run Conventional RAG Chain

In [31]:
import textwrap

question = "I am looking for a beginner drone. Please give me some recommendations."
resp = rag_chain.invoke(question)

print("\n\nRAG Response:\n")
print(textwrap.fill(resp, width=80))

Context Retrieved: 

Bundle:SkyHawk Zephyr Starter Package   Price: $129.99   Everything you need to
begin your drone journey:   This package includes the essentials to get you
flying and capturing stunning aerial footage. Perfect for first-time pilots who
want a user-friendly and reliable drone.   Package Includes:   SkyHawk Zephyr
Drone   Remote Controller   Rechargeable Battery   USB Charging Cable   Spare
Propellers   User Manual   Take flight with the SkyHawk Zephyr Starter Package!
Tags: ['Photography', 'Videography']


Product: SkyHawk Zephyr Drone   Price: $129.99   Weight: 220g (7.8 oz)   The
SkyHawk Zephyr is the perfect drone for beginners. It's built for effortless
flying, offering a smooth and enjoyable experience from the moment you unpack
it.   Features:   Simple Controls: Beginner friendly and intuitive controls,
plus automatic takeoff and landing.   Tough Build: Designed to handle rookie
mistakes, thanks to its robust construction.   Capture Memories: Record crisp HD
p

## Cleanup

> **USE WITH CAUTION!**

Clean up all the nodes/edges in your graph and remove your graph definition.

In [32]:
# Uncomment to clean up
# graph_store.cleanup()
# print("Property graph and all associated tables removed.")